# 135 · DeerFlow Research Skill

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/135-deerflow-research-skill/deerflow_research_skill_workbook.ipynb)

**What this notebook teaches:**
- What a DeerFlow skill is and how `SKILL.md` defines one
- The YAML front-matter fields that register a skill: `name`, `trigger`, `description`
- How trigger matching works — DeerFlow scans the message for `@skill-name` tokens
- How `subagent_enabled=True` + `plan_mode=True` cause the skill to delegate section drafting
- Where this fits in the DeerFlow example progression: 134 (raw HTTP) → 135 (custom skill) → 136 (SWE agent)

**Two halves:**
- **Part A (cells 1–6):** Pure Python — run anywhere. Read the real `SKILL.md`, understand the format, and simulate trigger dispatch.
- **Part B (cells 7–9):** Requires a running DeerFlow instance configured with the `course-research` skill.

| Example | DeerFlow mode | What you send | What DeerFlow does |
|---------|--------------|---------------|--------------------|
| 134 | Generic chat API | Any message | LLM + built-in tools |
| **135** | **Custom skill** | **`@course-research <query>`** | **Skill prompt + subagent delegation** |
| 136 | SWE agent | Code task | Sandboxed bash + file tools |

In [ ]:
# Part A · Cell 1 — Install
%pip install -q httpx python-dotenv

## Part A · What a DeerFlow Skill Is

A DeerFlow **skill** is a prompt module stored as `SKILL.md` in a named directory.  
DeerFlow reads skills at startup and registers each one by its `trigger` field.

The file has two parts:

**1. YAML front-matter** (between `---` delimiters) — the metadata DeerFlow needs to register the skill:

```yaml
---
name: course-research          # internal identifier, used in config.yaml
trigger: "@course-research"    # the @-mention that activates this skill
description: >                 # shown in skill listings / help output
  Generate a structured research report from uploaded thread materials.
---
```

**2. Markdown body** — the system-prompt injected when the skill fires:

```markdown
# Course Research Skill

You are a research assistant for the agent-framework course.

When activated with `@course-research <query>`:
1. Read all markdown files uploaded to the current thread workspace.
2. ...
```

**Why skills instead of raw prompts?**  
A skill is reusable across threads, versioned in the file system, and can specify  
subagent delegation rules in its markdown body — things you can't do in a one-off message.

In [ ]:
# Part A · Cell 2 — Print the actual SKILL.md from the filesystem
# This is runtime/skills/custom/course-research/SKILL.md in this example's directory.

import os
from pathlib import Path

# Locate SKILL.md relative to this notebook
NOTEBOOK_DIR = Path(os.getcwd())
# When run from examples/135-deerflow-research-skill/
SKILL_MD_CANDIDATES = [
    NOTEBOOK_DIR / "runtime" / "skills" / "custom" / "course-research" / "SKILL.md",
    NOTEBOOK_DIR / "examples" / "135-deerflow-research-skill" / "runtime" / "skills" / "custom" / "course-research" / "SKILL.md",
]

skill_path = next((p for p in SKILL_MD_CANDIDATES if p.exists()), None)

if skill_path:
    SKILL_CONTENT = skill_path.read_text()
    print(f"Read from: {skill_path}\n")
    print("=" * 60)
    print(SKILL_CONTENT)
    print("=" * 60)
else:
    # Fallback: use the known content so Part A still runs anywhere
    SKILL_CONTENT = """---
name: course-research
trigger: \"@course-research\"
description: >
  Generate a structured research report from uploaded thread materials.
---

# Course Research Skill

You are a research assistant for the agent-framework course.

When activated with `@course-research <query>`:

1. Read all markdown files uploaded to the current thread workspace.
2. Identify key concepts, implementation patterns, and design trade-offs.
3. Generate a structured report with these sections:
   - **Executive Summary** (2-3 sentences)
   - **Pattern Comparison** (table: Pattern | When to Use | Trade-offs)
   - **Implementation Notes** (bullet list of actionable details)
   - **Next Steps** (2-3 suggested follow-up examples from the course)

Write the final report as a markdown artifact named `research-report.md`.

Delegate section drafting to subagents when multiple source files are present:
- Subagent A: summarise patterns from all uploaded files
- Subagent B: draft the comparison table
- Supervisor: merge and finalise into `research-report.md`
"""
    print("[SKILL.md not found at expected path — using embedded copy]\n")
    print(SKILL_CONTENT)

In [ ]:
# Part A · Cell 3 — Parse the SKILL.md front-matter without a YAML library
# Shows exactly how DeerFlow extracts name, trigger, and description at startup.


def parse_skill_frontmatter(content: str) -> dict[str, str]:
    """Extract YAML front-matter fields from a SKILL.md string.
    Only handles the simple flat key: value format DeerFlow uses.
    """
    result: dict[str, str] = {}
    lines = content.splitlines()

    # Find the --- delimiters
    delimiters = [i for i, line in enumerate(lines) if line.strip() == "---"]
    if len(delimiters) < 2:
        return result

    front = lines[delimiters[0] + 1 : delimiters[1]]
    current_key = ""
    for line in front:
        if ": " in line and not line.startswith(" "):
            key, _, val = line.partition(": ")
            current_key = key.strip()
            result[current_key] = val.strip().strip('"')
        elif line.startswith("  ") and current_key:
            # Multi-line value (the description: > block)
            result[current_key] = (result.get(current_key, "") + " " + line.strip()).strip()
    return result


def extract_skill_body(content: str) -> str:
    """Return the markdown body after the front-matter block."""
    lines = content.splitlines()
    delimiters = [i for i, line in enumerate(lines) if line.strip() == "---"]
    if len(delimiters) < 2:
        return content
    return "\n".join(lines[delimiters[1] + 1 :]).strip()


meta = parse_skill_frontmatter(SKILL_CONTENT)
body = extract_skill_body(SKILL_CONTENT)

print("Front-matter fields:")
for k, v in meta.items():
    print(f"  {k:<14} = {v!r}")

print(f"\nBody length: {len(body)} chars")
print(f"Body preview: {body[:120]!r}...")

## Part A · How DeerFlow Discovers and Invokes Skills

At startup, DeerFlow scans each directory listed under `skills.custom` in `config.yaml`,  
reads the `SKILL.md`, and registers an entry: `trigger → (skill_body, config)`.

When a chat message arrives:

```
1. Tokenise the message on whitespace
2. For each token, check the skill registry for an exact match
   (triggers are @-mentions: "@course-research", "@code-review", etc.)
3. If a match is found:
   a. Inject the skill's markdown body as an extra system prompt
   b. Strip the trigger token from the user message
   c. Route to the LLM with the augmented system context
4. If subagent_enabled=True AND the skill body contains delegation instructions:
   DeerFlow spawns worker subagents for the delegated steps
5. If no skill matches: treat as a plain chat message
```

**Key insight:** The skill body is a prompt template, not code.  
All the delegation logic ("Subagent A: summarise...") is read by the LLM and acted on —  
DeerFlow doesn't parse the instructions itself.

In [ ]:
# Part A · Cell 4 — Simulate skill trigger matching
# Given a message, check if it activates the course-research skill.

SKILL_REGISTRY = {
    meta.get("trigger", "@course-research"): {
        "name": meta.get("name", "course-research"),
        "body": body,
        "description": meta.get("description", ""),
    }
}


def dispatch_skill(message: str, registry: dict) -> tuple[str | None, str]:
    """Return (skill_name, cleaned_message) if a trigger is found, else (None, message)."""
    tokens = message.split()
    for token in tokens:
        if token in registry:
            skill = registry[token]
            cleaned = " ".join(t for t in tokens if t != token).strip()
            return skill["name"], cleaned
    return None, message


test_messages = [
    "@course-research Generate a structured research report.",
    "@course-research Summarise the uploaded notes into a short report.",
    "What is the reactive loop pattern?",
    "@nonexistent-skill Do something.",
]

print("Trigger matching simulation:\n")
for msg in test_messages:
    skill_name, cleaned = dispatch_skill(msg, SKILL_REGISTRY)
    if skill_name:
        print(f"  INPUT : {msg!r}")
        print(f"  SKILL : {skill_name}")
        print(f"  QUERY : {cleaned!r}")
    else:
        print(f"  INPUT : {msg!r}")
        print(f"  SKILL : (none — plain chat)")
    print()

In [ ]:
# Part A · Cell 5 — Show how subagent delegation is described in the skill body
# Extracts the delegation section from the SKILL.md and explains the flow.

print("Skill body (what gets injected as system prompt):\n")
print(body)
print()

# Identify delegation instructions
delegation_lines = [
    line for line in body.splitlines()
    if "subagent" in line.lower() or "delegate" in line.lower()
]

print("Delegation instructions found in skill body:")
for line in delegation_lines:
    print(f"  {line}")

print()
print("When subagent_enabled=True, DeerFlow acts on these instructions:")
print("  1. Subagent A reads uploaded files and summarises patterns")
print("  2. Subagent B drafts the comparison table")
print("  3. Supervisor merges both into research-report.md")
print()
print("When subagent_enabled=False, the main LLM handles all steps itself.")

print("\nPart A complete — all cells run without a server.")

---
## Part B · Live Research Skill Run

The cells below require a running DeerFlow instance configured with the `course-research` skill.

```bash
# 1. Complete example 134's runtime setup first:
#    git clone https://github.com/bytedance/deer-flow ../deer-flow
#    cd ../deer-flow && pip install -e .

# 2. Copy the skill config and skills directory:
cp examples/135-deerflow-research-skill/runtime/config.sample.yaml ../deer-flow/conf/config.yaml
cp -r examples/135-deerflow-research-skill/runtime/skills ../deer-flow/

# 3. Start DeerFlow:
cd ../deer-flow && make dev
```

Cell 7 will raise immediately if DeerFlow is not reachable — **no silent hangs.**

In [ ]:
# Part B · Cell 6 — Connect (fail fast if DeerFlow is down)
import json
from dataclasses import dataclass, field
from typing import Iterator

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("DEERFLOW_BASE_URL", "http://localhost:8000")
THREAD_ID = "notebook-thread-135"

try:
    r = httpx.get(f"{BASE_URL}/api/health", timeout=5)
    r.raise_for_status()
    print(f"DeerFlow reachable at {BASE_URL}")
except Exception as exc:
    raise RuntimeError(
        f"DeerFlow is not reachable at {BASE_URL}.\n"
        "Start it with `make dev` in the deer-flow repo, then re-run this cell.\n"
        "Also ensure you copied runtime/config.sample.yaml and runtime/skills/ — see Part B header."
    ) from exc


# Reuse the ResearchRun class from src/workflow.py (inlined here for notebook clarity)
@dataclass
class ResearchRun:
    base_url: str
    thread_id: str
    _http: httpx.Client = field(
        default_factory=lambda: httpx.Client(
            timeout=httpx.Timeout(60.0, read=300.0)
        )
    )

    def upload(self, filename: str, content: str) -> str:
        resp = self._http.post(
            f"{self.base_url}/api/files/upload",
            files={"file": (filename, content.encode(), "text/markdown")},
            data={"thread_id": self.thread_id},
        )
        resp.raise_for_status()
        return resp.json().get("artifact_id", "unknown")

    def stream(
        self,
        prompt: str,
        *,
        plan_mode: bool = True,
        subagent_enabled: bool = True,
    ) -> Iterator[tuple[str, dict]]:
        with self._http.stream(
            "POST",
            f"{self.base_url}/api/chat/stream",
            json={
                "message": prompt,
                "thread_id": self.thread_id,
                "plan_mode": plan_mode,
                "subagent_enabled": subagent_enabled,
            },
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                raw = line.removeprefix("data:").strip()
                if raw == "[DONE]":
                    yield "end", {}
                    return
                try:
                    ev = json.loads(raw)
                    yield ev.get("type", "unknown"), ev
                except json.JSONDecodeError:
                    pass


run = ResearchRun(base_url=BASE_URL, thread_id=THREAD_ID)
print(f"ResearchRun ready  thread_id={THREAD_ID}")

In [ ]:
# Part B · Cell 7 — Upload source files and invoke the @course-research skill

SOURCE_FILES = {
    "01-state-graph.md": (
        "# State Graph Pattern\n"
        "Explicit directed graph. Nodes process state; edges route flow.\n"
        "Use when: pipeline has well-defined stages with predictable transitions.\n"
        "Trade-offs: verbose setup; great for auditability and replay.\n"
    ),
    "02-reactive-loop.md": (
        "# Reactive Loop Pattern\n"
        "Agent calls tools in a loop until a stop condition is met.\n"
        "Use when: task is open-ended; number of steps is unknown upfront.\n"
        "Trade-offs: harder to debug; powerful for code generation and research.\n"
    ),
    "03-multi-agent.md": (
        "# Multi-Agent Pattern\n"
        "Supervisor delegates sub-tasks to specialist worker agents.\n"
        "Use when: task can be parallelised or needs diverse domain expertise.\n"
        "Trade-offs: coordination overhead; scales well for deep research tasks.\n"
    ),
}

SKILL_PROMPT = (
    "@course-research Generate a structured research report: executive summary, "
    "pattern comparison table, implementation notes, and next-step recommendations."
)

_ICONS = {"message_chunk": "·", "tool_call": "⚙", "tool_result": "✓", "end": "■"}

print("Uploading source files ...")
for filename, content in SOURCE_FILES.items():
    artifact_id = run.upload(filename, content)
    print(f"  {filename} → {artifact_id}")

print(f"\nActivating skill (plan_mode=True, subagent_enabled=True) ...")
print(f"Prompt: {SKILL_PROMPT!r}\n")

live_events: list[tuple[str, dict]] = []
for et, data in run.stream(SKILL_PROMPT):
    live_events.append((et, data))
    icon = _ICONS.get(et, "?")
    if et == "message_chunk":
        print(data.get("content", ""), end="", flush=True)
    elif et == "tool_call":
        print(f"\n  {icon} tool_call: {data.get('tool_name', '?')}")
    elif et == "tool_result":
        snippet = str(data.get("content", ""))[:80]
        print(f"  {icon} tool_result: {snippet!r}")
    elif et == "end":
        print(f"\n  {icon} end")

In [ ]:
# Part B · Cell 8 — Summary: event counts, artifact check, and contrast table

counts: dict[str, int] = {}
for et, _ in live_events:
    counts[et] = counts.get(et, 0) + 1

print("── Event summary ──")
for et, n in sorted(counts.items()):
    print(f"  {_ICONS.get(et, '?')} {et:<18} {n}x")

# Look for a report artifact in tool_result events
artifact_hint = next(
    (
        str(data.get("content", ""))[:120]
        for et, data in live_events
        if et == "tool_result"
        and (".md" in str(data.get("content", "")) or "artifact" in str(data.get("content", "")).lower())
    ),
    None,
)

if artifact_hint:
    print(f"\n── Report artifact hint ──\n  {artifact_hint}")
else:
    print("\n── No artifact hint found in tool_result events (check streamed output above) ──")

print()
print("── How this example fits in the DeerFlow progression ──\n")
print("  134  raw HTTP      client.stream('any message')        LLM + built-in tools")
print("  135  custom skill  client.stream('@course-research …') skill prompt + subagents")
print("  136  SWE agent     agent.stream('fix this bug')        sandboxed bash + file tools")
print()
print(f"Thread : {THREAD_ID}")
print(f"Server : {BASE_URL}")
print(f"Skill  : {meta.get('name', 'course-research')} (trigger: {meta.get('trigger', '@course-research')})")